# Downloading data with an API

**Lecture 10 · Wednesday, 18 November 2026 · Notebook 1 of 3 (Video 10A)**

In Lecture 9 we cleaned a file someone had downloaded by hand. Here Python downloads the data itself, through an **API** (application programming interface): a service that returns data when a program asks for it. The download is **reproducible**: rerun the notebook and you get the newest numbers.

Our macro question: **how fast has the Danish economy grown, and how much of that came from working more hours rather than producing more per hour?** This is growth accounting, which you also meet in Macroeconomics I.

## How to work

| Recording | Notebook |
| --- | --- |
| Video 10A: Downloading data with an API | `02_From_API.ipynb` (this one) |
| Video 10B: Statistics within groups | `04_GroupBy.ipynb` |
| Video 10C: Combining datasets | `05_Merge.ipynb` |

`03_From_Excel.ipynb` is optional reference; its core was Video 9C. The questions for class are in **`Exercises.ipynb`**. **Before class (Tuesday 17 November, 23:59):** solve warm-ups **W1–W2** there, enter your answers in the Absalon quiz and upload the notebook.

**Needs an internet connection.** If Statistics Denmark does not respond (it can be slow when a whole class downloads at once), use the saved copy in Section 5.

**Table of contents**<a id='toc0_'></a>
- 1. [Find a table](#toc1_)
- 2. [Download one series and clean it](#toc2_)
- 3. [A function for repeated downloads](#toc3_)
- 4. [Analyse: growth, productivity and hours](#toc4_)
- 5. [Save a copy](#toc5_)
- 6. [Optional: data from FRED](#toc6_)

<a id='video-10a'></a>
> **Start Video 10A**  
> Question: how much of Danish GDP growth came from producing more per hour?

**Install once** (remove the `#` and run the cell, then put the `#` back). `dstapi` is a small package written by Alessandro Martinello for Statistics Denmark's API. Installing from GitHub needs **Git** on your computer; if the install or the download fails, load the saved copy in Section 5.

In [ ]:
# %pip install git+https://github.com/alemartinello/dstapi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dstapi import DstApi

## 1. <a id='toc1_'></a>[Find a table](#toc0_)

**Step 1** happens in the browser: search [statbank.dk](https://www.statbank.dk/) for a table. The national accounts are in table **NAN1**. In Python, `DstApi('NAN1')` connects to the table, and two methods describe it.

In [ ]:
NAN1 = DstApi('NAN1')
NAN1.tablesummary(language='en')

Each **variable** has codes. The unit (`PRISENHED`) matters: we want real GDP in **2020 prices** (`LAN_M`), not current prices.

In [ ]:
NAN1.variable_levels('PRISENHED', language='en')

In [ ]:
transactions = NAN1.variable_levels('TRANSAKT', language='en')
I = transactions['id'].isin(['B1GQK', 'EMPH_DC', 'EMPM_DC'])
transactions.loc[I]

So `B1GQK` is GDP, `EMPH_DC` total hours worked and `EMPM_DC` employment.

## 2. <a id='toc2_'></a>[Download one series and clean it](#toc0_)

A download is described by a dictionary: the table, and for each variable the codes we want. `'*'` means all values, here all years.

In [ ]:
params = {
    'table': 'NAN1',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'TRANSAKT', 'values': ['B1GQK']},     # GDP
        {'code': 'PRISENHED', 'values': ['LAN_M']},   # 2020 prices, bill. DKK
        {'code': 'Tid', 'values': ['*']},             # all years
    ],
}

raw = NAN1.get_data(params=params)
raw.head()

**Pause and predict:** what is wrong with this table before we can compute with it? Look at the column types.

In [ ]:
raw.info()

The values (`INDHOLD`) arrive as **text**, the years are not sorted, and the column names are in Danish. The cleaning steps are the ones from Lecture 9:

In [ ]:
gdp = raw.copy()
gdp['INDHOLD'] = gdp['INDHOLD'].astype(float)                 # text -> number
gdp = gdp.rename(columns={'TID': 'year', 'INDHOLD': 'real_gdp'})
gdp = gdp.drop(columns=['TRANSAKT', 'PRISENHED'])
gdp = gdp.sort_values('year').set_index('year')               # the year becomes the row label
gdp.head()

`set_index('year')` makes the year the row label, so `gdp.loc[2020]` gives the value for 2020.

## 3. <a id='toc3_'></a>[A function for repeated downloads](#toc0_)

We need three series. Instead of copying the code three times, we put it in a function (Lecture 3).

In [ ]:
def load_NAN1(transakt, prisenhed, varname):
    """Download one series from NAN1 and return it with the year as index."""

    params = {
        'table': 'NAN1',
        'format': 'BULK',
        'lang': 'en',
        'variables': [
            {'code': 'TRANSAKT', 'values': [transakt]},
            {'code': 'PRISENHED', 'values': [prisenhed]},
            {'code': 'Tid', 'values': ['*']},
        ],
    }

    df = DstApi('NAN1').get_data(params=params)
    df['INDHOLD'] = df['INDHOLD'].astype(float)
    df = df.rename(columns={'TID': 'year', 'INDHOLD': varname})
    df = df.drop(columns=['TRANSAKT', 'PRISENHED'])
    return df.sort_values('year').set_index('year')


df = load_NAN1('B1GQK', 'LAN_M', 'real_gdp')             # bill. DKK, 2020 prices
df['hours'] = load_NAN1('EMPH_DC', 'V_M', 'hours')['hours']              # million hours
df['employed'] = load_NAN1('EMPM_DC', 'V_M', 'employed')['employed']     # thousand persons
df.tail()

Adding a column from another DataFrame works because both have the **same index** (the year): `pandas` lines up the rows by year.

## 4. <a id='toc4_'></a>[Analyse: growth, productivity and hours](#toc0_)

**Labour productivity** is GDP per hour worked. GDP is in billion DKK and hours in million hours, so multiplying by 1,000 gives DKK per hour.

In [ ]:
df['productivity'] = df['real_gdp']/df['hours']*1000     # DKK per hour, 2020 prices

df['gdp_growth'] = df['real_gdp'].pct_change()*100        # percent per year
df[['real_gdp', 'productivity', 'gdp_growth']].tail()

`pct_change()` compares each row with the **previous** row: $100\times(x_t/x_{t-1}-1)$. The first year has no previous year, so it is `NaN`.

Which years had falling GDP?

In [ ]:
recession = df['gdp_growth'] < 0
print('Years with falling real GDP:', df.index[recession].to_list())

fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(1, 1, 1)
ax.bar(df.index, df['gdp_growth'])
ax.axhline(0.0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Real GDP growth (%)')
ax.set_title('Danish GDP growth')
fig.tight_layout()
plt.show()

### Growth accounting

Because GDP = productivity × hours, **log growth** splits exactly into two parts:

$$\Delta\log(\text{GDP})=\Delta\log(\text{productivity})+\Delta\log(\text{hours}).$$

`np.log(...)` takes logs of a whole column, and `.diff()` takes the change from the previous year. A known identity is a good test of our code:

In [ ]:
dlog = pd.DataFrame({
    'gdp': np.log(df['real_gdp']).diff()*100,
    'productivity': np.log(df['productivity']).diff()*100,
    'hours': np.log(df['hours']).diff()*100,
})
print('Identity holds:', np.allclose(dlog['gdp'].iloc[1:], dlog['productivity'].iloc[1:] + dlog['hours'].iloc[1:]))

**Pause and predict:** over the whole period, did Danish growth come mainly from more hours or from more output per hour?

In [ ]:
print('Average annual growth, percent (log points):')
print(dlog.mean().round(2))

In [ ]:
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(1, 1, 1)
ax.plot(df.index, df['productivity']/df['productivity'].iloc[0]*100, label='Productivity (GDP per hour)')
ax.plot(df.index, df['hours']/df['hours'].iloc[0]*100, label='Total hours worked')
ax.set_xlabel('Year')
ax.set_ylabel(f'Index, {df.index[0]} = 100')
ax.set_title('Productivity and hours worked')
ax.legend()
fig.tight_layout()
plt.show()

Compare the two lines and the averages above: which part of growth dominates? In Macroeconomics I, the Solow model explains productivity growth by capital accumulation and technological progress.

## 5. <a id='toc5_'></a>[Save a copy](#toc0_)

Save the cleaned data, so the rest of your work does not depend on the API responding. If the download fails in class, load the saved copy instead.

In [ ]:
df.to_csv('data/NAN1_growth.csv')

# if the API does not respond, use the saved copy instead:
# df = pd.read_csv('data/NAN1_growth.csv', index_col='year')

**Summary:** find the table and its codes → download with a `params` dictionary → clean (types, names, sort, index) → check (plots, a known identity) → save.

> **Stop Video 10A.** Next: statistics within groups, in `04_GroupBy.ipynb`.

## 6. <a id='toc6_'></a>[Optional: data from FRED](#toc0_)

**Optional reading.** US and international data are in [FRED](https://fred.stlouisfed.org/), the database of the Federal Reserve Bank of St. Louis. The `fredapi` package works much like `dstapi`, but you first need a free personal **API key**:

1. create an account and request a key at [fredaccount.stlouisfed.org](https://fredaccount.stlouisfed.org/apikeys);
2. save the key in a text file **outside** your project folder, never in a notebook you share;
3. then:

```python
from fredapi import Fred
with open('path/to/fredapi.txt') as f:
    fred = Fred(api_key=f.read().strip())
us_gdp = fred.get_series('GDPCA')     # US real GDP, annual
```

### References

[Statistics Denmark API](https://www.dst.dk/en/Statistik/hjaelp-til-statistikbanken/api), [dstapi](https://github.com/alemartinello/dstapi), [fredapi](https://github.com/mortada/fredapi).